In [1]:
import os 
import numpy as np
import pandas as pd
import nrrd

In [2]:
from pathlib import Path

current = Path().resolve()

while not (current / ".git").exists():
    current = current.parent

PROJECT_ROOT = current

This notebook shows how did we create the all_celltypes_composition_with_unassigned_regions.csv

In [3]:
celltype_folder = PROJECT_ROOT / "extension/data/"

#Load me-types
file_path = os.path.join(celltype_folder, "brain__with_unassigned_regions_me_types_composition.csv")
me_data_df = pd.read_csv(file_path, index_col=0, header=[0, 1])
# Combine the two levels of the MultiIndex into one
me_data_df.columns = [f"{col[0]}|{col[1]}" for col in me_data_df.columns]

#Load e-types
file_path = os.path.join(celltype_folder, "brain__with_unassigned_regions_e_types_composition.csv")
e_data_df = pd.read_csv(file_path, index_col=0)


#Load m-types
file_path = os.path.join(celltype_folder, "brain__with_unassigned_regions_m_types_composition.csv")
m_data_df = pd.read_csv(file_path, index_col=0)

data_df = pd.concat(
    [e_data_df, m_data_df, me_data_df, ],
    axis=1
)

#Load region ids from the Allen Institute so we can use the common coordinate framework for visualisation
file_path = PROJECT_ROOT / "densities_app/parcellation_to_parcellation_term_membership_extend.csv"

parcellation_annotation = pd.read_csv(
    file_path,
    usecols=["cluster_as_filename", "label_numbers"],
    index_col="cluster_as_filename"
).sort_index()

#Trim Allen data from unused region ids, keep only those which are present in the CCFv3
ids = parcellation_annotation[parcellation_annotation.index.isin(data_df.index)]

#Merge label numbers with cell type data, remove duplicates which come from non-leaf-regions having the same names as leaf-regions
combined_df = pd.merge(
    data_df,
    ids,
    left_index=True,
    right_index=True
)

combined_df = combined_df[~combined_df.index.duplicated(keep='first')]
combined_df.to_csv(os.path.join(celltype_folder, "all_celltypes_composition_with_unassigned_regions.csv"))
combined_df.shape

(719, 530)